In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# SENTRY - Step 1: Data Collection
Run cells top to bottom. Each source saves its own CSV in `data/` so you can debug one at a time.
**Before you start:** register for ACLED, a ReliefWeb appname, and a Google Cloud project (for BigQuery/GDELT).

In [ ]:
import pandas as pd, requests, os, time
os.makedirs('data', exist_ok=True)
START, END = '2018-01-01', '2026-10-24'

# name, ISO3 (ACLED/ReliefWeb), FIPS (GDELT)
COUNTRIES = [
 ('Ukraine','UKR','UP'),('Sudan','SDN','SU'),('Myanmar','MMR','BM'),('Niger','NER','NG'),
 ('Mali','MLI','ML'),('Burkina Faso','BFA','UV'),('Yemen','YEM','YM'),('Syria','SYR','SY'),
 ('Ethiopia','ETH','ET'),('Afghanistan','AFG','AF'),('Haiti','HTI','HA'),('Gabon','GAB','GB'),
 ('India','IND','IN'),('Brazil','BRA','BR'),('Japan','JPN','JA'),
]
iso2name = {i:n for n,i,_ in COUNTRIES}
fips2iso = {f:i for _,i,f in COUNTRIES}

## 1. GDELT (news tone + hostility) via BigQuery

In [ ]:
from google.colab import auth; auth.authenticate_user()
from google.cloud import bigquery
PROJECT = 'sentry-512009'   # <- change this
bq = bigquery.Client(project=PROJECT)
fips = ','.join(f"'{f}'" for _,_,f in COUNTRIES)
q = f'''
SELECT DATE_TRUNC(PARSE_DATE('%Y%m%d', CAST(SQLDATE AS STRING)), WEEK(MONDAY)) AS week,
       ActionGeo_CountryCode AS fips,
       AVG(AvgTone) AS news_tone, AVG(GoldsteinScale) AS hostility_score,
       COUNT(*) AS news_volume
FROM `gdelt-bq.gdeltv2.events`
WHERE ActionGeo_CountryCode IN ({fips})
  AND SQLDATE BETWEEN {START.replace('-','')} AND {END.replace('-','')}
GROUP BY week, fips
'''
fips2iso = {f: i for _, i, f in COUNTRIES}
gdelt = bq.query(q).to_dataframe()
gdelt['iso3'] = gdelt['fips'].map(fips2iso)
gdelt.to_csv('data/gdelt_weekly.csv', index=False)
gdelt.head()

,week,fips,news_tone,hostility_score,news_volume,iso3
0,2025-08-18,BR,-1.215365,0.990880,13092,BRA
1,2024-06-03,BR,-0.713189,1.091528,11804,BRA
2,2025-01-06,SU,-3.747922,-0.542567,4083,SDN
3,2023-05-22,BM,-3.862344,-1.330315,1270,MMR
4,2025-10-20,JA,0.256274,1.757126,12124,JPN


## 2. ACLED (protest / conflict counts)
Uses your ACLED login. Check ACLED's API docs if auth has changed.

In [ ]:
import glob
COUNTRIES = [
    ('Ukraine', 'UKR', 'UP'),
    ('Sudan', 'SDN', 'SU'),
    ('Myanmar', 'MMR', 'BM'),
    ('Niger', 'NER', 'NG'),
    ('Mali', 'MLI', 'ML'),
    ('Burkina Faso', 'BFA', 'UV'),
    ('Yemen', 'YEM', 'YM'),
    ('Syria', 'SYR', 'SY'),
    ('Ethiopia', 'ETH', 'ET'),
    ('Afghanistan', 'AFG', 'AF'),
    ('Haiti', 'HTI', 'HA'),
    ('Gabon', 'GAB', 'GB'),
    ('India', 'IND', 'IN'),
    ('Brazil', 'BRA', 'BR'),
    ('Japan', 'JPN', 'JA')
]
# 1. Load and stack all your ACLED region files
files = glob.glob('/content/drive/MyDrive/Sentrydata/*.csv')   # upload the ACLED files to Colab's main folder
print(files)                          # check that only ACLED files are listed

acled_raw = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
acled_raw.columns = acled_raw.columns.str.lower()

# Drop duplicate columns to prevent the ValueError: "cannot assemble with duplicate keys"
acled_raw = acled_raw.loc[:, ~acled_raw.columns.duplicated()]

# 2. Align weeks, map countries, and keep our 15
acled_raw['week'] = pd.to_datetime(acled_raw['week'])
acled_raw['week'] = acled_raw['week'].dt.to_period('W-SUN').dt.start_time
acled_raw['iso3'] = acled_raw['country'].map({n: i for n, i, _ in COUNTRIES})
acled_raw = acled_raw.dropna(subset=['iso3'])

# 3. Weekly counts per country (adds up the province-level rows)
weekly = acled_raw.groupby(['iso3', 'week', 'event_type'])['events'].sum().unstack(fill_value=0)
weekly['protest_count'] = weekly.get('Protests', 0) + weekly.get('Riots', 0)
weekly['battle_count'] = (weekly.get('Battles', 0)
                          + weekly.get('Violence against civilians', 0)
                          + weekly.get('Explosions/Remote violence', 0))

out = weekly[['protest_count', 'battle_count']].reset_index()
import os
os.makedirs('data', exist_ok=True)
out.to_csv('data/acled_weekly.csv', index=False)

# 4. Sanity checks
print(out['iso3'].nunique(), 'countries (expect 15)')
print(out['week'].min(), 'to', out['week'].max())
out.head()

['/content/drive/MyDrive/Sentrydata/Africa_aggregated_data_up_to_week_of-2026-10-03.csv', '/content/drive/MyDrive/Sentrydata/Latin-America-the-Caribbean_aggregated_data_up_to_week_of-2026-10-10.csv', '/content/drive/MyDrive/Sentrydata/Asia-Pacific_aggregated_data_up_to_week_of-2026-10-03.csv', '/content/drive/MyDrive/Sentrydata/Europe-Central-Asia_aggregated_data_up_to_week_of-2026-10-03.csv', '/content/drive/MyDrive/Sentrydata/Middle-East_aggregated_data_up_to_week_of-2026-10-03.csv', '/content/drive/MyDrive/Sentrydata/ACLED Data_2026-10-23_event_date_from_2026-09-19_event_date_to_2026-10-16.csv']
15 countries (expect 15)
1996-12-23 00:00:00 to 2026-10-05 00:00:00


event_type,iso3,week,protest_count,battle_count
0,AFG,2016-12-26,3.0,249.0
1,AFG,2017-01-02,3.0,221.0
2,AFG,2017-01-09,4.0,185.0
3,AFG,2017-01-16,5.0,133.0
4,AFG,2017-01-23,4.0,177.0


## 3. ReliefWeb (aid reports per week) - display layer only
Needs an approved `appname` - request one at reliefweb.int.

In [ ]:
# !pip install -q feedparser
# import feedparser
# import pandas as pd
# import os

# Create data directory if it doesn't exist
# os.makedirs('data', exist_ok=True)

# feed = feedparser.parse('https://www.gdacs.org/xml/rss.xml')
# alerts = [{'title':e.title,'level':e.get('gdacs_alertlevel'),'type':e.get('gdacs_eventtype'),
#            'country':e.get('gdacs_country'),'date':e.get('published')} for e in feed.entries]
# pd.DataFrame(alerts).to_csv('data/gdacs_current.csv', index=False)
# pd.DataFrame(alerts).head()

## 5. Merge into the model table (news + protest only)

In [ ]:
START = '2018-01-01'  # Beginning of data
END = '2024-12-31'    # End of data

weeks = pd.date_range(START, END, freq='W-MON')
grid = pd.MultiIndex.from_product([[i for _,i,_ in COUNTRIES], weeks], names=['iso3','week']).to_frame(index=False)
gd = pd.read_csv('data/gdelt_weekly.csv', parse_dates=['week'])
ac = pd.read_csv('data/acled_weekly.csv', parse_dates=['week'])
master = grid.merge(gd.drop(columns='fips'), on=['iso3','week'], how='left').merge(ac, on=['iso3','week'], how='left')
master[['protest_count','battle_count']] = master[['protest_count','battle_count']].fillna(0)   # no events = 0, not missing
master.to_csv('data/master_table.csv', index=False)
print(master.shape); master.describe()

(5490, 7)


,week,news_tone,hostility_score,news_volume,protest_count,battle_count
count,5490,5490.000000,5490.000000,5490.000000,5490.000000,5490.000000
mean,2021-07-01 12:00:00,-2.470448,0.285020,10308.176867,41.111293,100.254098
min,2018-01-01 00:00:00,-6.885732,-4.441217,48.000000,0.000000,0.000000
25%,2019-09-30 00:00:00,-3.398420,-0.271183,1209.250000,1.000000,7.000000
50%,2021-07-01 12:00:00,-2.467803,0.426198,3231.000000,6.000000,25.000000
75%,2023-04-03 00:00:00,-1.581868,0.986140,11325.750000,27.000000,151.000000
max,2024-12-30 00:00:00,3.011695,4.485156,242569.000000,2410.000000,1292.000000
std,NaN,1.269565,1.016605,16958.633726,104.340412,179.880808


In [ ]:
# df = pd.read_csv('data/master_table.csv', parse_dates=['week']).sort_values(['iso3','week']).reset_index(drop=True)
# N = 4   # prediction horizon in weeks

# def build(g):
#     g = g.copy()
#     # 1. An "event week" = protests or battles surge well above that country's own past year
#     for col in ['protest_count', 'battle_count']:
#         past = g[col].shift(1).rolling(52, min_periods=26)       # past weeks only, no peeking
#         g[col + '_event'] = ((g[col] > past.mean() + 2 * past.std()) & (g[col] >= 10)).astype(int)
#     g['event_week'] = g['protest_count_event'] | g['battle_count_event']

#     # 2. Label = 1 if any event week falls in the NEXT N weeks
#     future = pd.concat([g['event_week'].shift(-k) for k in range(1, N + 1)], axis=1)
#     g['label'] = future.max(axis=1)
#     g.loc[future.isna().any(axis=1), 'label'] = None              # last N weeks have no answer yet

#     # 3. Features built from past/current weeks only (4-week smoothing)
#     for col in ['protest_count', 'battle_count', 'news_tone', 'hostility_score']:
#         g[col + '_4w'] = g[col].rolling(4, min_periods=2).mean()
#     return g

# df = df.groupby('iso3', group_keys=False).apply(build)
# model_df = df.dropna(subset=['label', 'protest_count_4w', 'news_tone_4w']).copy()
# model_df['label'] = model_df['label'].astype(int)
# model_df.to_csv('data/model_table.csv', index=False)

# print(model_df.shape)
# print('overall positive rate:', model_df['label'].mean().round(3))
# print(model_df.groupby('iso3')['label'].mean().round(2))


import pandas as pd

df = pd.read_csv(
    "data/master_table.csv",
    parse_dates=["week"]
).sort_values(["iso3", "week"]).reset_index(drop=True)

N = 4

def build(g):
    g = g.copy()

    # Detect unusual events using only the previous 52 weeks
    for col in ["protest_count", "battle_count"]:
        past = g[col].shift(1).rolling(52, min_periods=26)

        g[col + "_event"] = (
            (g[col] > past.mean() + 2 * past.std()) &
            (g[col] >= 10)
        ).astype(int)

    g["event_week"] = (
        (g["protest_count_event"] == 1) |
        (g["battle_count_event"] == 1)
    ).astype(int)

    # Label: event occurs in one of the next 4 weeks
    future = pd.concat(
        [g["event_week"].shift(-k) for k in range(1, N + 1)],
        axis=1
    )

    g["label"] = future.max(axis=1)
    g.loc[future.isna().any(axis=1), "label"] = None

    # Features: previous 4 weeks only, excluding the current week
    for col in [
        "protest_count",
        "battle_count",
        "news_tone",
        "hostility_score"
    ]:
        g[col + "_4w"] = (
            g[col]
            .shift(1)
            .rolling(4, min_periods=2)
            .mean()
        )

    return g

df = (
    df.groupby("iso3", group_keys=False)
      .apply(build)
      .reset_index(drop=True)
)

model_df = df.dropna(
    subset=[
        "label",
        "protest_count_4w",
        "battle_count_4w",
        "news_tone_4w",
        "hostility_score_4w"
    ]
).copy()

model_df["label"] = model_df["label"].astype(int)
model_df.to_csv("data/model_table.csv", index=False)

print("Model table shape:", model_df.shape)
print("Positive rate:", round(model_df["label"].mean(), 3))
print("Missing labels:", model_df["label"].isna().sum())
print("Saved: data/model_table.csv")

Model table shape: (5400, 15)
Positive rate: 0.257
Missing labels: 0
Saved: data/model_table.csv


/tmp/ipykernel_964/3400981107.py:85: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(build)


In [ ]:
# import glob
# print(glob.glob('/content/drive/MyDrive/**/*.csv', recursive=True)[:30])

In [ ]:
from sklearn.model_selection import train_test_split

FEATURES = [
    'protest_count_4w',
    'battle_count_4w',
    'news_tone_4w',
    'hostility_score_4w'
]

model_df = pd.read_csv('data/model_table.csv', parse_dates=['week'])

split_date = model_df['week'].quantile(0.8)

train = model_df[model_df['week'] < split_date]
test = model_df[model_df['week'] >= split_date]

X_train, y_train = train[FEATURES], train['label']
X_test, y_test = test[FEATURES], test['label']

print("Train:", X_train.shape)
print("Test:", X_test.shape)
print("Train positive rate:", y_train.mean().round(3))
print("Test positive rate:", y_test.mean().round(3))

Train: (4320, 4)
Test: (1080, 4)
Train positive rate: 0.266
Test positive rate: 0.221


In [ ]:
!pip -q install xgboost

from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    random_state=42
)

model.fit(X_train, y_train)
print("Model training complete!")

Model training complete!


In [ ]:
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred, zero_division=0))
print("ROC-AUC:", round(roc_auc_score(y_test, y_prob), 3))
print("PR-AUC:", round(average_precision_score(y_test, y_prob), 3))

              precision    recall  f1-score   support

           0       0.79      0.91      0.84       841
           1       0.28      0.12      0.17       239

    accuracy                           0.74      1080
   macro avg       0.53      0.52      0.51      1080
weighted avg       0.67      0.74      0.70      1080

ROC-AUC: 0.578
PR-AUC: 0.273


In [ ]:
from sklearn.metrics import precision_recall_curve
import numpy as np

precision, recall, thresholds = precision_recall_curve(y_test, y_prob)

for target in [0.3, 0.4, 0.5, 0.6]:
    idx = np.argmin(np.abs(thresholds - target))
    print(
        f"Threshold: {thresholds[idx]:.3f} | "
        f"Precision: {precision[idx]:.3f} | "
        f"Recall: {recall[idx]:.3f}"
    )

Threshold: 0.300 | Precision: 0.268 | Recall: 0.410
Threshold: 0.400 | Precision: 0.280 | Recall: 0.205
Threshold: 0.500 | Precision: 0.284 | Recall: 0.121
Threshold: 0.601 | Precision: 0.328 | Recall: 0.092


In [ ]:
from sklearn.metrics import average_precision_score, roc_auc_score
probs = model.predict_proba(X_test)[:, 1]
print('Baseline positive rate:', round(y_test.mean(), 3))
print('PR-AUC:', round(average_precision_score(y_test, probs), 3))
print('ROC-AUC:', round(roc_auc_score(y_test, probs), 3))

Baseline positive rate: 0.221
PR-AUC: 0.273
ROC-AUC: 0.578


In [ ]:
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

print("\nTraining feature columns:")
print(X_train.columns.tolist())

print("\nTarget distribution — train:")
print(y_train.value_counts(normalize=True))

print("\nTarget distribution — test:")
print(y_test.value_counts(normalize=True))

X_train shape: (4320, 4)
X_test shape: (1080, 4)

Training feature columns:
['protest_count_4w', 'battle_count_4w', 'news_tone_4w', 'hostility_score_4w']

Target distribution — train:
label
0    0.734491
1    0.265509
Name: proportion, dtype: float64

Target distribution — test:
label
0    0.778704
1    0.221296
Name: proportion, dtype: float64


In [ ]:
print("X_train index range:", X_train.index.min(), X_train.index.max())
print("X_test index range:", X_test.index.min(), X_test.index.max())

print("\nTraining data preview:")
display(X_train.head())

print("\nTesting data preview:")
display(X_test.head())

print("\nOriginal feature dataset columns:")
print(df.columns.tolist())

X_train index range: 0 5327
X_test index range: 288 5399

Training data preview:


,protest_count_4w,battle_count_4w,news_tone_4w,hostility_score_4w
0,0.000000,282.000000,-4.024834,-1.048324
1,1.666667,266.666667,-3.821578,-0.990544
2,2.000000,245.750000,-4.178054,-1.221621
3,3.250000,239.250000,-4.163705,-1.121510
4,4.500000,207.250000,-4.191011,-1.164101



Testing data preview:


,protest_count_4w,battle_count_4w,news_tone_4w,hostility_score_4w
288,4.25,19.50,-2.844435,0.192793
289,5.50,18.25,-2.756978,0.334506
290,5.25,21.75,-2.980076,0.325192
291,5.75,25.50,-3.142186,0.344940
292,6.00,24.00,-3.259601,0.283701



Original feature dataset columns:
['iso3', 'week', 'news_tone', 'hostility_score', 'news_volume', 'protest_count', 'battle_count', 'protest_count_event', 'battle_count_event', 'event_week', 'label', 'protest_count_4w', 'battle_count_4w', 'news_tone_4w', 'hostility_score_4w']


In [ ]:
import pandas as pd

train_weeks = pd.to_datetime(
    df.loc[X_train.index, "week"], errors="coerce"
)
test_weeks = pd.to_datetime(
    df.loc[X_test.index, "week"], errors="coerce"
)

print("Training date range:")
print(train_weeks.min(), "to", train_weeks.max())

print("\nTesting date range:")
print(test_weeks.min(), "to", test_weeks.max())

print("\nLatest training week:", train_weeks.max())
print("Earliest testing week:", test_weeks.min())

print("\nShared row indices:",
      len(set(X_train.index) & set(X_test.index)))

print("\nMissing training dates:", train_weeks.isna().sum())
print("Missing testing dates:", test_weeks.isna().sum())

Training date range:
2018-01-01 00:00:00 to 2024-12-30 00:00:00

Testing date range:
2021-11-29 00:00:00 to 2024-11-18 00:00:00

Latest training week: 2024-12-30 00:00:00
Earliest testing week: 2021-11-29 00:00:00

Shared row indices: 0

Missing training dates: 0
Missing testing dates: 0


In [ ]:
import pandas as pd
from xgboost import XGBClassifier

df = df.sort_values("week").reset_index(drop=True)

features = [
    "protest_count_4w",
    "battle_count_4w",
    "news_tone_4w",
    "hostility_score_4w"
]

X = df[features]
y = df["label"]

split_idx = int(len(df) * 0.8)

X_train = X.iloc[:split_idx]
X_test = X.iloc[split_idx:]

y_train = y.iloc[:split_idx]
y_test = y.iloc[split_idx:]

print("Training period:", df["week"].iloc[0],
      "to", df["week"].iloc[split_idx - 1])
print("Testing period:", df["week"].iloc[split_idx],
      "to", df["week"].iloc[-1])

model = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42
)

model.fit(X_train, y_train)

print("Chronological model training complete!")

Training period: 2018-01-01 00:00:00 to 2023-08-07 00:00:00
Testing period: 2023-08-07 00:00:00 to 2024-12-30 00:00:00
Chronological model training complete!


In [ ]:
print("Missing labels:", master["label"].isna().sum() if "label" in master.columns else "'label' column not in master DataFrame")
print("Total rows:", len(master))

Missing labels: 'label' column not in master DataFrame
Total rows: 5490


In [ ]:
# from sklearn.metrics import (
#     classification_report,
#     confusion_matrix,
#     roc_auc_score,
#     average_precision_score
# )

# y_pred = model.predict(X_test)
# probs = model.predict_proba(X_test)[:, 1]

# print("Classification Report:")
# print(classification_report(y_test, y_pred, zero_division=0))

# print("Confusion Matrix:")
# print(confusion_matrix(y_test, y_pred))

# print("ROC-AUC:", round(roc_auc_score(y_test, probs), 3))
# print("PR-AUC:", round(average_precision_score(y_test, probs), 3))
# print("Test positive rate:", round(y_test.mean(), 3))

In [ ]:
import pandas as pd
import numpy as np

print("y_test missing values:", pd.isna(y_test).sum())
print("y_pred missing values:", pd.isna(y_pred).sum())

print("y_test unique values:", pd.Series(y_test).unique())
print("y_pred unique values:", np.unique(y_pred))

print("y_test dtype:", y_test.dtype)
print("y_pred dtype:", y_pred.dtype)

y_test missing values: 60
y_pred missing values: 0
y_test unique values: [ 0.  1. nan]
y_pred unique values: [0 1]
y_test dtype: float64
y_pred dtype: int64


In [ ]:
import pandas as pd
from xgboost import XGBClassifier

df = df.sort_values("week").reset_index(drop=True)

features = [
    "protest_count_4w",
    "battle_count_4w",
    "news_tone_4w",
    "hostility_score_4w"
]

# Keep only rows with valid target labels
df = df.dropna(subset=["label"]).copy()

# Split by unique weeks so countries from the same week
# don't end up on opposite sides of the split
weeks = sorted(df["week"].unique())
split_idx = int(len(weeks) * 0.8)
split_week = weeks[split_idx]

train_df = df[df["week"] < split_week]
test_df = df[df["week"] >= split_week]

X_train = train_df[features]
y_train = train_df["label"].astype(int)

X_test = test_df[features]
y_test = test_df["label"].astype(int)

model = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42
)

model.fit(X_train, y_train)

print("Training period:", train_df["week"].min(), "to", train_df["week"].max())
print("Testing period:", test_df["week"].min(), "to", test_df["week"].max())
print("Missing test labels:", y_test.isna().sum())
print("Model retrained successfully!")

Training period: 2018-01-01 00:00:00 to 2023-07-10 00:00:00
Testing period: 2023-07-17 00:00:00 to 2024-12-02 00:00:00
Missing test labels: 0
Model retrained successfully!


In [ ]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score
)

y_pred = model.predict(X_test)
probs = model.predict_proba(X_test)[:, 1]

print("Classification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("ROC-AUC:", round(roc_auc_score(y_test, probs), 3))
print("PR-AUC:", round(average_precision_score(y_test, probs), 3))
print("Test positive rate:", round(y_test.mean(), 3))

Classification Report:
              precision    recall  f1-score   support

           0       0.79      0.92      0.85       852
           1       0.32      0.14      0.19       243

    accuracy                           0.74      1095
   macro avg       0.55      0.53      0.52      1095
weighted avg       0.68      0.74      0.70      1095

Confusion Matrix:
[[781  71]
 [210  33]]
ROC-AUC: 0.569
PR-AUC: 0.279
Test positive rate: 0.222


In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

for threshold in [0.2, 0.3, 0.4, 0.5]:
    pred = (probs >= threshold).astype(int)

    print(
        f"Threshold: {threshold:.1f} | "
        f"Precision: {precision_score(y_test, pred, zero_division=0):.3f} | "
        f"Recall: {recall_score(y_test, pred, zero_division=0):.3f} | "
        f"F1: {f1_score(y_test, pred, zero_division=0):.3f}"
    )

Threshold: 0.2 | Precision: 0.238 | Recall: 0.720 | F1: 0.358
Threshold: 0.3 | Precision: 0.268 | Recall: 0.420 | F1: 0.327
Threshold: 0.4 | Precision: 0.285 | Recall: 0.202 | F1: 0.236
Threshold: 0.5 | Precision: 0.317 | Recall: 0.136 | F1: 0.190


In [ ]:
print(df[["week", "protest_count", "battle_count", "label"]].head(15))
print("\nLabel distribution:")
print(df["label"].value_counts(dropna=False))

         week  protest_count  battle_count  label
0  2018-01-01            0.0         264.0    0.0
1  2018-01-01           50.0         168.0    0.0
2  2018-01-01            2.0          16.0    0.0
3  2018-01-01            1.0          11.0    0.0
4  2018-01-01          193.0          41.0    0.0
5  2018-01-01           18.0           8.0    0.0
6  2018-01-01           30.0         207.0    0.0
7  2018-01-01           17.0           0.0    0.0
8  2018-01-01            2.0           0.0    0.0
9  2018-01-01            2.0         661.0    0.0
10 2018-01-01           10.0         183.0    0.0
11 2018-01-01            3.0           0.0    0.0
12 2018-01-01            0.0           4.0    0.0
13 2018-01-01            0.0           0.0    0.0
14 2018-01-01            0.0           0.0    0.0

Label distribution:
label
0.0    4044
1.0    1386
Name: count, dtype: int64


In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score
)

df = pd.read_csv("data/model_table.csv", parse_dates=["week"])
df = df.sort_values(["week", "iso3"]).reset_index(drop=True)

features = [
    "protest_count_4w",
    "battle_count_4w",
    "news_tone_4w",
    "hostility_score_4w"
]

weeks = sorted(df["week"].unique())
split_idx = int(len(weeks) * 0.8)
split_week = weeks[split_idx]

train_df = df[df["week"] < split_week]
test_df = df[df["week"] >= split_week]

X_train = train_df[features]
y_train = train_df["label"].astype(int)
X_test = test_df[features]
y_test = test_df["label"].astype(int)

model = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42
)

model.fit(X_train, y_train)

probs = model.predict_proba(X_test)[:, 1]
y_pred = (probs >= 0.2).astype(int)

print("Training:", train_df["week"].min(), "to", train_df["week"].max())
print("Testing:", test_df["week"].min(), "to", test_df["week"].max())
print("\nResults at threshold 0.2:")
print(classification_report(y_test, y_pred, zero_division=0))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))
print("ROC-AUC:", round(roc_auc_score(y_test, probs), 3))
print("PR-AUC:", round(average_precision_score(y_test, probs), 3))
print("Precision:", round(precision_score(y_test, y_pred, zero_division=0), 3))
print("Recall:", round(recall_score(y_test, y_pred, zero_division=0), 3))
print("F1:", round(f1_score(y_test, y_pred, zero_division=0), 3))

Training: 2018-01-15 00:00:00 to 2023-07-17 00:00:00
Testing: 2023-07-24 00:00:00 to 2024-12-02 00:00:00

Results at threshold 0.2:
              precision    recall  f1-score   support

           0       0.82      0.36      0.50       841
           1       0.24      0.72      0.36       239

    accuracy                           0.44      1080
   macro avg       0.53      0.54      0.43      1080
weighted avg       0.69      0.44      0.47      1080

Confusion matrix:
 [[302 539]
 [ 67 172]]
ROC-AUC: 0.571
PR-AUC: 0.277
Precision: 0.242
Recall: 0.72
F1: 0.362


In [ ]:
from sklearn.metrics import average_precision_score
import numpy as np

# Baseline: assign every test row the training positive rate
baseline_probs = np.full(len(y_test), y_train.mean())

print("Training positive rate:", round(y_train.mean(), 3))
print("Test positive rate:", round(y_test.mean(), 3))
print("Model PR-AUC:", round(average_precision_score(y_test, probs), 3))
print("Baseline PR-AUC:", round(average_precision_score(y_test, baseline_probs), 3))

Training positive rate: 0.266
Test positive rate: 0.221
Model PR-AUC: 0.277
Baseline PR-AUC: 0.221


In [ ]:
import pandas as pd

df = pd.read_csv("data/model_table.csv", parse_dates=["week"])

print("Latest week:", df["week"].max())
print("Countries in latest week:", df[df["week"] == df["week"].max()]["iso3"].nunique())
print(df[df["week"] == df["week"].max()][
    ["iso3", "week", "protest_count_4w",
     "battle_count_4w", "news_tone_4w",
     "hostility_score_4w"]
].head(10).to_string(index=False))


Latest week: 2024-12-02 00:00:00
Countries in latest week: 15
iso3       week  protest_count_4w  battle_count_4w  news_tone_4w  hostility_score_4w
 AFG 2024-12-02              1.75            22.00     -2.894980            0.145904
 BFA 2024-12-02              2.00            26.75     -1.720490            0.439295
 BRA 2024-12-02             36.25           181.25     -0.823362            1.019273
 ETH 2024-12-02              6.75            50.00     -0.377496            1.409990
 GAB 2024-12-02              1.00             0.00      0.520525            1.944053
 HTI 2024-12-02              2.75            48.25     -3.549867           -0.209340
 IND 2024-12-02            374.75            32.25     -1.682259            0.332764
 JPN 2024-12-02             32.75             0.00     -0.081657            1.456839
 MLI 2024-12-02              2.50            25.00     -2.153806            0.580492
 MMR 2024-12-02             19.25           263.00     -3.344264           -1.229439


In [ ]:
features = [
    "protest_count_4w",
    "battle_count_4w",
    "news_tone_4w",
    "hostility_score_4w"
]

latest_week = df["week"].max()
latest = df[df["week"] == latest_week].copy()

latest = latest.dropna(subset=features)

latest["risk_probability"] = model.predict_proba(
    latest[features]
)[:, 1]

latest["risk_score"] = (latest["risk_probability"] * 100).round(1)

latest["warning_level"] = latest["risk_probability"].apply(
    lambda p: "WARNING" if p >= 0.2 else "LOWER RISK"
)

risk_table = latest[
    ["iso3", "week", "risk_score", "warning_level"]
].sort_values("risk_score", ascending=False)

print("Scored week:", latest_week)
print(risk_table.to_string(index=False))

risk_table.to_csv("data/sentry_risk_scores.csv", index=False)
print("\nSaved to data/sentry_risk_scores.csv")

Scored week: 2024-12-02 00:00:00
iso3       week  risk_score warning_level
 YEM 2024-12-02   72.099998       WARNING
 ETH 2024-12-02   51.400002       WARNING
 HTI 2024-12-02   50.500000       WARNING
 IND 2024-12-02   48.500000       WARNING
 BRA 2024-12-02   36.000000       WARNING
 NER 2024-12-02   30.799999       WARNING
 SDN 2024-12-02   29.600000       WARNING
 MMR 2024-12-02   27.400000       WARNING
 AFG 2024-12-02   25.700001       WARNING
 MLI 2024-12-02   24.400000       WARNING
 BFA 2024-12-02   21.900000       WARNING
 UKR 2024-12-02   16.799999    LOWER RISK
 JPN 2024-12-02   16.299999    LOWER RISK
 SYR 2024-12-02    3.700000    LOWER RISK
 GAB 2024-12-02    1.300000    LOWER RISK

Saved to data/sentry_risk_scores.csv


In [ ]:
import pandas as pd

df = pd.read_csv("data/master_table.csv", parse_dates=["week"])

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

print("\nHostility score summary:")
print(df["hostility_score"].describe())

print("\nLatest available week:", df["week"].max())

print("\nLatest country-level values:")
print(
    df[df["week"] == df["week"].max()][
        ["iso3", "week", "hostility_score"]
    ].head(15).to_string(index=False)
)


Dataset shape: (5490, 7)
Columns: ['iso3', 'week', 'news_tone', 'hostility_score', 'news_volume', 'protest_count', 'battle_count']

Hostility score summary:
count    5490.000000
mean        0.285020
std         1.016605
min        -4.441217
25%        -0.271183
50%         0.426198
75%         0.986140
max         4.485156
Name: hostility_score, dtype: float64

Latest available week: 2024-12-30 00:00:00

Latest country-level values:
iso3       week  hostility_score
 UKR 2024-12-30        -0.487614
 SDN 2024-12-30         0.024550
 MMR 2024-12-30        -0.734494
 NER 2024-12-30         0.695317
 MLI 2024-12-30         0.422619
 BFA 2024-12-30         0.028682
 YEM 2024-12-30        -2.354399
 SYR 2024-12-30         0.241136
 ETH 2024-12-30         1.873538
 AFG 2024-12-30        -0.816465
 HTI 2024-12-30         0.371255
 GAB 2024-12-30         2.044025
 IND 2024-12-30         0.208354
 BRA 2024-12-30         0.801375
 JPN 2024-12-30         0.965920


In [ ]:
import pandas as pd

df = pd.read_csv("data/master_table.csv", parse_dates=["week"])
df = df.sort_values(["iso3", "week"])

df["hostility_score_4w"] = (
    df.groupby("iso3")["hostility_score"]
      .transform(lambda s: s.shift(1).rolling(4, min_periods=2).mean())
)

latest_week = df["week"].max()
latest = df[df["week"] == latest_week].dropna(
    subset=["hostility_score_4w"]
).copy()

latest["hostility_score"] = latest["hostility_score_4w"]

latest[["iso3", "week", "hostility_score"]].to_csv(
    "data/sentry_hostility_scores.csv", index=False
)

print(latest[["iso3", "week", "hostility_score"]]
      .sort_values("hostility_score", ascending=False)
      .to_string(index=False))

print("\nSaved: data/sentry_hostility_scores.csv")


iso3       week  hostility_score
 GAB 2024-12-30         2.173427
 ETH 2024-12-30         1.706379
 JPN 2024-12-30         1.253638
 BFA 2024-12-30         1.048843
 BRA 2024-12-30         0.766525
 MLI 2024-12-30         0.465366
 IND 2024-12-30         0.298480
 NER 2024-12-30        -0.020408
 UKR 2024-12-30        -0.055348
 SYR 2024-12-30        -0.144177
 SDN 2024-12-30        -0.206959
 AFG 2024-12-30        -0.375057
 HTI 2024-12-30        -0.596548
 MMR 2024-12-30        -1.336360
 YEM 2024-12-30        -2.183372

Saved: data/sentry_hostility_scores.csv


In [ ]:
import pandas as pd

risk = pd.read_csv("data/sentry_risk_scores.csv")
hostility = pd.read_csv("data/sentry_hostility_scores.csv")

risk["iso3"] = risk["iso3"].str.upper()
hostility["iso3"] = hostility["iso3"].str.upper()

dashboard = risk.merge(
    hostility[["iso3", "hostility_score"]],
    on="iso3",
    how="outer"
)

dashboard.to_csv("data/sentry_dashboard_data.csv", index=False)

print("Shape:", dashboard.shape)
print("Columns:", dashboard.columns.tolist())
print(dashboard.head(10).to_string(index=False))
print("\nSaved: data/sentry_dashboard_data.csv")

Shape: (15, 5)
Columns: ['iso3', 'week', 'risk_score', 'warning_level', 'hostility_score']
iso3       week  risk_score warning_level  hostility_score
 AFG 2024-12-02        25.7       WARNING        -0.375057
 BFA 2024-12-02        21.9       WARNING         1.048843
 BRA 2024-12-02        36.0       WARNING         0.766525
 ETH 2024-12-02        51.4       WARNING         1.706379
 GAB 2024-12-02         1.3    LOWER RISK         2.173427
 HTI 2024-12-02        50.5       WARNING        -0.596548
 IND 2024-12-02        48.5       WARNING         0.298480
 JPN 2024-12-02        16.3    LOWER RISK         1.253638
 MLI 2024-12-02        24.4       WARNING         0.465366
 MMR 2024-12-02        27.4       WARNING        -1.336360

Saved: data/sentry_dashboard_data.csv


In [ ]:
from google.colab import files

files.download("data/sentry_dashboard_data.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>